In [ ]:
import weaviate
from langchain_core.documents import Document
from pathlib import Path
import sys 
import pandas as pd
import ast
import re

# Langchain intergrated LLM, embedding
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Testset generation by graph
from ragas.testset.graph import KnowledgeGraph
from ragas.testset.graph import Node, NodeType

# Evaluate 
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas.metrics import ResponseRelevancy, Faithfulness
from ragas import evaluate
from ragas.testset.transforms import default_transforms, default_transforms_for_prechunked, apply_transforms

# Built modules

%load_ext autoreload
%autoreload 2
# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root)) 
from utils import parse_contexts, clean_hop_markers

from src.config import get_settings
from src.rag_workflow import AgenticRAG
from src.retriever import retrieve
from src.config import get_settings
setting = get_settings()

client = weaviate.connect_to_local(
    port=8080,
    grpc_port=50051
)

collection = client.collections.get("TestCollection")


In [ ]:
docs = []
for doc in collection.iterator():
    doc = doc.properties
    docs.append(Document(page_content=doc.get('text'), metadata={
        'type': doc.get('type', ''),
        'id': doc.get('element_id', ''),
        'source': doc.get('source', ''),
        'image_path': doc.get('image_path', ''),
        'page_number': doc.get('page_number', '')
    }))

In [ ]:
len(docs)

In [ ]:

llm_generator = LangchainLLMWrapper(ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature= 0.4,
    top_p=0.8,
    google_api_key=setting.gemini_api_key
))

embedding_generator = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-001',
    task_type='retrieval_document',
    google_api_key=setting.gemini_api_key
))


# 1. Create graph of documents

In [ ]:
kg = KnowledgeGraph()

In [ ]:
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.CHUNK,
            properties={'page_content': doc.page_content, 'document_metadata': doc.metadata}
        )
    )

## 1.1 Enrich document graph

In [ ]:
trans = default_transforms_for_prechunked(
    llm=llm_generator,
    embedding_model=embedding_generator
)

apply_transforms(kg, trans)

In [ ]:
generator = TestsetGenerator(llm=llm_generator, embedding_model=embedding_generator)
generator.knowledge_graph = kg
dataset = generator.generate(testset_size=20)

In [ ]:
dataset.to_pandas().head(5)

In [ ]:
dataset.to_pandas().to_csv('testset.csv', index=False)

In [ ]:
df_testset = dataset.to_pandas()

In [ ]:
df_testset['reference_contexts'][18]

# 2. Clean 'reference_contexts' column

In [ ]:
test_set = pd.read_csv('testset.csv')

# Parse and clean reference_contexts
test_set['reference_contexts_parsed'] = test_set['reference_contexts'].apply(parse_contexts)
test_set['reference_contexts'] = test_set['reference_contexts_parsed'].apply(clean_hop_markers)

In [ ]:
test_set['reference_contexts'][19]

# 3. Add 'retrieved_contexts' and 'response' column to test_set 

In [ ]:
# Get retrieved_contexts, rewritten_queries, and response
rag = AgenticRAG()

retrieved_contexts_list = []
rewritten_queries_list = []
response_list = []

for idx, row in test_set.iterrows():
    result = rag.graph.invoke(
        {
            'query': row['user_input'],
            'collection_name': "TestCollection",
        },
        config={'configurable': {'thread_id': idx}}
    )
    retrieved_docs = result.get('retrieved_documents', [])
    # Format list of retrieved documents 
    retrieved_contexts_list.append([doc.properties.get('text', '') for doc in retrieved_docs])
    # Get rewritten queries from result
    rewritten_queries_list.append(result.get('queries', []))
    # Get AI response (last message)
    response_list.append(result['messages'][-1].content)

# Assign columns at once (not inside the loop)
test_set['retrieved_contexts'] = retrieved_contexts_list
test_set['rewritten_queries'] = rewritten_queries_list
test_set['response'] = response_list

In [ ]:
test_set.drop('reference_contexts_parsed', axis=1, inplace=True)

In [ ]:
test_set.to_csv('testset_processed.csv', index=False)

In [ ]:
test_set = pd.read_csv('testset_processed.csv')
test_set.head(5)

In [ ]:
test_set['reference_contexts'][0]

# 4. Calculate exact recall and precision

In [ ]:
from utils import exact_precision, exact_recall

test_set['retrieved_contexts'] = test_set['retrieved_contexts'].apply(parse_contexts)
test_set['reference_contexts'] = test_set['reference_contexts'].apply(parse_contexts)

In [ ]:
precision_scores = []
recall_scores = []

for idx, sample in test_set.iterrows():
    retrieved = sample['retrieved_contexts']
    reference = sample['reference_contexts']

    precision_scores.append(exact_precision(retrieved, reference))
    recall_scores.append(exact_recall(retrieved, reference))

test_set['exact_precision'] = precision_scores
test_set['exact_recall'] = recall_scores

In [ ]:
test_set['exact_recall']

In [ ]:
test_set['exact_precision']

In [ ]:
test_set.to_csv('testset_processed.csv', index=False)

In [ ]:
retrieval_evaluation_csv = pd.read_csv('testset_processed.csv')

In [ ]:
import numpy as np
np.mean(retrieval_evaluation_csv['exact_precision'])

In [ ]:
retrieval_evaluation_columns = ['user_input', 'reference_contexts', 'retrieved_contexts', 'exact_precision', 'exact_recall']

retrieval_evaluation_csv = retrieval_evaluation_csv[retrieval_evaluation_columns]

In [ ]:
retrieval_evaluation_csv.drop(['query_style', 'query_length', 'persona_name'], axis=1, inplace=True)

In [ ]:
retrieval_evaluation_csv

In [ ]:
import matplotlib.pyplot as plt

# Create figure with 2 separate subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Histogram of Precision
axes[0].hist(retrieval_evaluation_csv['exact_precision'], bins=10, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Retrieval Precision')
axes[0].set_xlim(0, 1)

# 2. Histogram of Recall
axes[1].hist(retrieval_evaluation_csv['exact_recall'], bins=10, alpha=0.7, color='coral', edgecolor='black')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Retrieval Recall')
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
retrieval_evaluation_csv.to_csv('retrieval_evaluation.csv', index=False)

## 4.1 Observations and analysis:

1. Precision is low ~ 0.2 (4 cases got 0) since might retrieve too many documents <- query rewriter generates many irrelevant queries.

2. Recall is medium ~ 0.67 (4 cases got 0) <- low K results that not included relevant docs

3. There are many cases that low precision (<0.5) and high recall (1) <- successfully retrievel doc to answer but also irrelevent docs

4. There are many cases that both 0 precision and recall (problem)


## 4.2 Conclusion
- Prioritize RECALL (target~0.8) than PRECISION (target~0.5):
    + Since RECALL must high -> get complete relevant docs. PRECISION could low -> irrelevent docs -> cost tokens (acceptable)




## 4.3 Action 

- Improve RECALL:
    1. Query rewriter: LLM-as-a-judge
    2. Retriever: top_k, alpha, chunking strategy


### 4.3.1 Error analysis

In [ ]:
retrieval_eval_df = pd.read_csv("retrieval_evaluation.csv")

In [ ]:
retrieval_eval_df.drop('reference', axis=1, inplace=True)

**Inspect on samples got score 0**

In [ ]:
zeros_df = retrieval_eval_df.loc[(retrieval_eval_df['exact_recall'] == 0)]

In [ ]:
zeros_df

In [ ]:
zeros_df.to_csv('error_retrieval_df.csv', index=False)

In [ ]:
# Inspect query rewriter results
for idx, row in zeros_df.iterrows():
    print(row['user_input'])
    print(row['rewritten_queries'])
    print('-' * 50)

### 4.3.2 Evaluate Query rewriter and fine-tune the prompt

In [ ]:
from utils import evaluate_rewrites
evaluator = ChatGoogleGenerativeAI(
    model='gemini-2.5-pro',
    temperature=0.1,
    google_api_key=setting.gemini_api_key
)
eval_df = evaluate_rewrites(
    original_queries=queries,
    rewritten_queries=rewritten_queries,
    llm=evaluator
)

In [ ]:
for text in eval_df['reasoning']:
    print(text)
    print('-' * 50)

**=> Query rewriter works well -> move to retriever**

In [ ]:
rewritten_queries = []
rag = AgenticRAG()
for query in zeros_df['user_input']:
    state = {
        'query': query,
        'collection_name': "TestCollection",
    }
    config={'configurable': {'thread_id': query}}

    result = rag.query_rewriter(state)
    rewritten_queries.append(result['queries'])

In [ ]:
from utils import systematic_retrieval_eval

queries = zeros_df['user_input'].to_list()

reference_contexts = [parse_contexts(ctx) for ctx in zeros_df['reference_contexts']]

finetuned_df = systematic_retrieval_eval(
    queries=queries,
    reference_contexts=reference_contexts,
    top_k=15,
    top_reranker=5,
    rewritten_queries=rewritten_queries
)

In [ ]:
finetuned_df['rewritten_queries'] = rewritten_queries

In [ ]:
finetuned_df

In [ ]:
print('Mean recall: ', finetuned_df['recall_scores'].mean())
print('Mean precision: ', finetuned_df['precision_scores'].mean())

**Observations:**
- test 4 got improved: recall 0 -> 0.5 -> finetune prompt is effective
- BUT others still bad, all 0 recall
- Rewriter does not works well <- retriever good at semantic but extract optimized keyword + split query cause dilutes result pool

**Next action:**
- Tune prompt that just clean queries, not paraphase, keep original as much as can be, ONLY decompose when needed

In [ ]:
rewritten_queries = []
rag = AgenticRAG()
for query in zeros_df['user_input']:
    state = {
        'query': query,
        'collection_name': "TestCollection",
    }
    config={'configurable': {'thread_id': f'{query}2'}}

    result = rag.query_rewriter(state)
    rewritten_queries.append(result['queries'])

In [ ]:
finetuned_df = systematic_retrieval_eval(
    queries=queries,
    reference_contexts=reference_contexts,
    top_k=15,
    top_reranker=5,
    rewritten_queries=rewritten_queries
)


In [ ]:
finetuned_df

In [ ]:
print('Mean recall: ', finetuned_df['recall_scores'].mean())
print('Mean precision: ', finetuned_df['precision_scores'].mean())

In [ ]:
finetuned_df['rewritten_queries'] = rewritten_queries
finetuned_df.to_csv('finetuned_rewriter_df.csv', index=False)

In [ ]:
finetuned_df['rewritten_queries'][1]

In [ ]:
for text in finetuned_df['reference_contexts'][1]:
    print(text)
    print('-'*50)

In [ ]:
for text in finetuned_df['retrieved_contexts'][1]:
    print(text)
    print('-'*50)

### 4.3.3 Improvement experiment 2: Increase top_k of hybrid search

In [ ]:
exp_2 = systematic_retrieval_eval(
    queries=queries, 
    reference_contexts=reference_contexts, 
    top_k=50,
    top_reranker=3
)
print('Mean recall: ', exp_2['recall_scores'].mean())
print('Mean precision: ', exp_2['precision_scores'].mean())

CONCLUSION: Nothing change. Even retrieve more docs but they're considered to be irrelevant <- low score 

### 4.3.4 Improvement experiment 3: Increase top_k of reranker

In [ ]:
exp_3 = systematic_retrieval_eval(
    queries=queries, 
    reference_contexts=reference_contexts, 
    top_k=15,
    top_reranker=5
)
print('Mean recall: ', exp_3['recall_scores'].mean())
print('Mean precision: ', exp_3['precision_scores'].mean())

In [ ]:
exp_3.to_csv('without_rewriter_df.csv', index=False)

CONCLUSION: 0.45 recall and 0.15 precision with intial search = 15 and rerank = 5

In [ ]:
exp_3_2 = systematic_retrieval_eval(
    queries=queries, 
    reference_contexts=reference_contexts, 
    top_k=15,
    top_reranker=5,
    rewritten_queries=rewritten_queries
)
print('Mean recall: ', exp_3_2['recall_scores'].mean())
print('Mean precision: ', exp_3_2['precision_scores'].mean())

In [ ]:
exp_3_2

CONCLUSION: worse than original query -> evaluate full test set

### 4.3.5 Evaluate all test set

In [ ]:
rewritten_queries = []
rag = AgenticRAG()
for query in retrieval_eval_df['user_input']:
    state = {
        'query': query,
        'collection_name': "TestCollection",
    }
    config={'configurable': {'thread_id': f'{query}5'}}

    result = rag.query_rewriter(state)
    rewritten_queries.append(result['queries'])

In [ ]:
# improvement experiment 1: disable 'query rewriter'

queries = retrieval_eval_df['user_input'].to_list()
reference_contexts = [parse_contexts(ctx) for ctx in retrieval_eval_df['reference_contexts']]


test_all_df = systematic_retrieval_eval(
    queries=queries, 
    reference_contexts=reference_contexts, 
    top_k=15,
    top_reranker=5,
    rewritten_queries=rewritten_queries
)

print('Mean recall: ', test_all_df['recall_scores'].mean())
print('Mean precision: ', test_all_df['precision_scores'].mean())

In [ ]:
test_all_df_without = systematic_retrieval_eval(
    queries=queries, 
    reference_contexts=reference_contexts, 
    top_k=15,
    top_reranker=5,
    # rewritten_queries=rewritten_queries
)

print('Mean recall: ', test_all_df_without['recall_scores'].mean())
print('Mean precision: ', test_all_df_without['precision_scores'].mean())

**Conclusion:** 

Max performance: 

    - Mean recall:  0.6904761904761905

    - Mean precision:  0.1656462585034014

With fine-tuned rewriter prompt + tune hyperparameters: top_k = 15, top_rerank = 5

=> Still low -> improve chunking strategy since the reference contexts still low score (irrelevant) in retrieved docs

### 4.3.6 Evaluate collection with new chunking strategy, enriched multimodal chunks

In [ ]:
import weaviate
from langchain_core.documents import Document
from pathlib import Path
import sys
import pandas as pd
import ast
import re
import numpy as np

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas.testset import TestsetGenerator
from ragas.testset.transforms import default_transforms_for_prechunked, apply_transforms

%load_ext autoreload
%autoreload 2
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from utils import parse_contexts, clean_hop_markers, exact_precision, exact_recall

from src.config import get_settings
from src.rag_workflow import AgenticRAG
setting = get_settings()

client = weaviate.connect_to_local(port=8080, grpc_port=50051)
collection = client.collections.get("NewChunkingCollection")

In [ ]:
docs = []
for doc in collection.iterator():
    doc = doc.properties
    docs.append(Document(page_content=doc.get('text'), metadata={
        'type': doc.get('type', ''),
        'id': doc.get('element_id', ''),
        'source': doc.get('source', ''),
        'image_path': doc.get('image_path', ''),
        'page_number': doc.get('page_number', '')
    }))
print(f"Loaded {len(docs)} documents from NewChunkingCollection")

In [ ]:
llm_generator = LangchainLLMWrapper(ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.4,
    top_p=0.8,
    google_api_key=setting.gemini_api_key
))

embedding_generator = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-001',
    task_type='retrieval_document',
    google_api_key=setting.gemini_api_key
))

In [ ]:
kg = KnowledgeGraph()

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.CHUNK,
            properties={'page_content': doc.page_content, 'document_metadata': doc.metadata}
        )
    )
print(f"Added {len(kg.nodes)} nodes to knowledge graph")

In [ ]:
trans = default_transforms_for_prechunked(
    llm=llm_generator,
    embedding_model=embedding_generator
)

apply_transforms(kg, trans)

In [ ]:
generator = TestsetGenerator(llm=llm_generator, embedding_model=embedding_generator)
generator.knowledge_graph = kg
dataset = generator.generate(testset_size=20)

dataset.to_pandas().to_csv('testset_new_chunking.csv', index=False)
print(f"Generated {len(dataset)} test samples")
dataset.to_pandas().head()

In [ ]:
from src.rag_workflow import AgenticRAG

rag = AgenticRAG()
rewritten_queries = []
for idx, row in test_set.iterrows():
    # Call rewriter node directly, skip retrieval & generation
    print(f"Row {idx}: {row['queries']}")
    state = {'query': row['queries'], 'messages': []}
    result = rag.query_rewriter(state)
    rewritten_queries.append(result['queries'])
    print('-'*50)


In [ ]:
import asyncio
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference

context_recall_scorer = NonLLMContextRecall()
context_precision_scorer = NonLLMContextPrecisionWithReference()

recall_scores = []
precision_scores = []

for idx, row in result_df.iterrows():
    sample = SingleTurnSample(
        retrieved_contexts=row['retrieved_contexts'],
        reference_contexts=row['reference_contexts'],
    )
    recall = await context_recall_scorer.single_turn_ascore(sample)
    precision = await context_precision_scorer.single_turn_ascore(sample)

    recall_scores.append(recall)
    precision_scores.append(precision)
    print(f"Row {idx}: precision={precision:.4f}, recall={recall:.4f}")

test_set['nonllm_precision'] = precision_scores
test_set['nonllm_recall'] = recall_scores

print(f"\nMean NonLLM Precision: {np.mean(precision_scores):.4f}")
print(f"Mean NonLLM Recall:    {np.mean(recall_scores):.4f}")

# test_set.to_csv('new_chunking_retrieval_evaluation.csv', index=False)

## Final Observation on Retrieval evaluation
1. Improvements:

    a. Ingestion:

        - Chunking smaller (3000 char -> 1500 char -> better retrieval)

        - Enrich images, tables with their captions

    b. Retrieval

        - Increase top k (15 -> 25), reranker (3 -> 7), alpha 0.5

        - Give Mean Precision@7: 0.5786, Mean Recall@7:    0.8413


In [ ]:
from src.rag_workflow import AgenticRAG
rag = AgenticRAG()

retrieved_contexts_list = []
rewritten_queries_list = []
response_list = []

for idx, row in test_set.iterrows():
    rag = AgenticRAG()
    result = rag.graph.invoke(
        {
            'query': row['user_input'],
            'collection_name': "NewChunkCollection",
        },
        config={'configurable': {'thread_id': f"{idx}gts"}}
    )
    retrieved_docs = result.get('retrieved_documents', [])
    retrieved_contexts_list.append([doc.properties.get('text', '') for doc in retrieved_docs])
    rewritten_queries_list.append(result.get('queries', []))
    response_list.append(result['messages'][-1].content)


test_set['retrieved_contexts'] = retrieved_contexts_list
test_set['rewritten_queries'] = rewritten_queries_list
test_set['response'] = response_list
test_set.to_csv('best_retrieval_performance.csv', index=False)

In [ ]:
import weaviate
from langchain_core.documents import Document
from pathlib import Path
import sys
import pandas as pd
import ast
import re
import numpy as np

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset
from ragas.testset import TestsetGenerator
from ragas.testset.transforms import default_transforms_for_prechunked, apply_transforms

%load_ext autoreload
%autoreload 2
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from utils import parse_contexts, clean_hop_markers, exact_precision, exact_recall

from src.config import get_settings
from src.rag_workflow import AgenticRAG
setting = get_settings()

client = weaviate.connect_to_local(port=8080, grpc_port=50051)
collection = client.collections.get("NewChunkingCollection")

from src.rag_workflow import AgenticRAG

rag = AgenticRAG()
rewritten_queries = []
test_set = pd.read_csv('data/new_chunking_retrieval_evaluation.csv')
test_set['reference_contexts'] = test_set['reference_contexts'].apply(parse_contexts)

for idx, row in test_set.iterrows():
    # Call rewriter node directly, skip retrieval & generation
    # print(f"Row {idx}: {row['queries']}")
    state = {'query': row['user_input'], 'messages': []}
    result = rag.query_rewriter(state)
    rewritten_queries.append(result['queries'])
    # print('-'*50)

from utils import systematic_retrieval_eval
result_df = systematic_retrieval_eval(
    queries=test_set['user_input'].tolist(),
    reference_contexts=test_set['reference_contexts'].tolist(),
    top_k=25,
    top_reranker=7,
    rewritten_queries=rewritten_queries,
    collection_name='NewChunkCollection'
)

import asyncio
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import NonLLMContextRecall, NonLLMContextPrecisionWithReference

context_recall_scorer = NonLLMContextRecall()
context_precision_scorer = NonLLMContextPrecisionWithReference()

recall_scores = []
precision_scores = []

for idx, row in result_df.iterrows():
    sample = SingleTurnSample(
        retrieved_contexts=row['retrieved_contexts'],
        reference_contexts=row['reference_contexts'],
    )
    recall = await context_recall_scorer.single_turn_ascore(sample)
    precision = await context_precision_scorer.single_turn_ascore(sample)

    recall_scores.append(recall)
    precision_scores.append(precision)
    print(f"Row {idx}: precision={precision:.4f}, recall={recall:.4f}")

test_set['nonllm_precision'] = precision_scores
test_set['nonllm_recall'] = recall_scores

print(f"\nMean NonLLM Precision: {np.mean(precision_scores):.4f}")
print(f"Mean NonLLM Recall:    {np.mean(recall_scores):.4f}")

# test_set.to_csv('new_chunking_retrieval_evaluation.csv', index=False)

In [ ]:
test_set['rewritten_queries'] = rewritten_queries
test_set['retrieved_contexts'] = result_df['retrieved_contexts']
test_set['exact_precision'] = result_df['precision_scores']
test_set['exact_recall'] = result_df['recall_scores']
test_set.to_csv('best_retrieval_performance.csv', index=False)

In [ ]:
test_set

# 5. Generation evaluation  

In [ ]:
evaluator_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.1,
    google_api_key=setting.gemini_api_key
))
evaluator_embedding = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-001',
    google_api_key=setting.gemini_api_key
))

In [ ]:
from ragas import evaluate
from ragas.metrics import ResponseRelevancy, Faithfulness
samples = []
for idx, row in test_set.iterrows():
    sample = SingleTurnSample(
        user_input=row['user_input'],
        response=row['response'],
        retrieved_contexts=row['retrieved_contexts']
    )
    samples.append(sample)

eval_dataset = EvaluationDataset(samples=samples)

gen_results = evaluate(
    dataset=eval_dataset,
    metrics=[
        ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embedding),
        Faithfulness(llm=evaluator_llm),
    ],
    llm=evaluator_llm,
    embeddings=evaluator_embedding
)

In [ ]:
gen_results.to_pandas()

In [ ]:
gen_results

In [ ]:
gen_results_csv = gen_results.to_pandas()

In [ ]:
gen_results_csv.to_csv('generation_evaluation.csv', index=False)

## Observations on Generation: 

- Generation evaluation (ResponseRelevancy, Faithfulness):

    - Faithfulness ~ 0.92: the generation stick well to retrieved docs and less hallcinations 
    
    - ResponseRelevancy ~ 0.81: The response is highly relevant to the query

